In [102]:
dir = "/Users/abhinavmohanty/Documents/Python/DataCamp/Data Scientist/Projects/"
nobel = "A Visual History of Nobel Prize Winners/datasets/nobel.csv"
cosmetics = "Comparing Cosmetics by Ingredients/datasets/cosmetics.csv"
disney = "Disney Movies and Box Office Success/datasets/disney_movies_total_gross.csv"
lefthand = "Do Left-handed People Really Die Young_/datasets/iris.csv"
monthly_deaths = "Dr. Semmelweis and the Discovery of Handwashing/datasets/monthly_deaths.csv"
yearly_deaths = "Dr. Semmelweis and the Discovery of Handwashing/datasets/yearly_deaths_by_clinic.csv"
colors = "Exploring the History of Lego/datasets/colors.csv"
sets = "Exploring the History of Lego/datasets/sets.csv"
movies = "Find Movie Similarity from Plot Summaries/datasets/movies.csv"
keywords = "Generating Keywords for Google Ads/keywords.csv"
transfusion = "Give Life_ Predict Blood Donations/datasets/transfusion.data"
temperature = "Introduction to DataCamp Projects/datasets/global_temperature.csv"
cookie = "Mobile Games AB Testing with Cookie Cats/datasets/cookie_cats.csv"
cc = "Predicting Credit Card Approvals/datasets/cc_approvals.data"
social_us = "Real-time Insights from Social Media Data/datasets/USTrends.json"
social_ww = "Real-time Insights from Social Media Data/datasets/WWTrends.json"
dates = "Recreating John Snow's Ghost Map/datasets/dates.csv"
dates = "Recreating John Snow's Ghost Map/datasets/dates.csv"
deaths = "Recreating John Snow's Ghost Map/datasets/deaths.csv"
pumps = "Recreating John Snow's Ghost Map/datasets/pumps.csv"
pumps_deaths = "Recreating John Snow's Ghost Map/datasets/deaths_and_pumps.csv"
benchmark = "Risk and Returns_ The Sharpe Ratio/datasets/benchmark_data.csv"
iris = "Risk and Returns_ The Sharpe Ratio/datasets/iris.csv"
stock = "Risk and Returns_ The Sharpe Ratio/datasets/stock_data.csv"
apps = "The Android App Market on Google Play/datasets/apps.csv"
reviews = "The Android App Market on Google Play/datasets/user_reviews.csv"

file_path = dir + nobel
print(file_path)

/Users/abhinavmohanty/Documents/Python/DataCamp/Data Scientist/Projects/A Visual History of Nobel Prize Winners/datasets/nobel.csv


In [2]:
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder.master( "local[3]" ) \
    .appName('Spark') \
    .getOrCreate()

In [4]:
print("App Name:", spark.sparkContext.appName)
#print("Master:" spark.sparkContext.master)

App Name: Spark


In [63]:
from pyspark.sql.functions import size, col, split, year, month, dayofmonth, sum, count, round, max, min, avg, lead, lag, rank, dense_rank, first, last

In [6]:
from pyspark.sql import Window

In [13]:
## Nobel

In [7]:
df_nobel = spark.read.csv(file_path, header=True, inferSchema=True)
df_nobel.printSchema()

root
 |-- year: integer (nullable = true)
 |-- category: string (nullable = true)
 |-- prize: string (nullable = true)
 |-- motivation: string (nullable = true)
 |-- prize_share: string (nullable = true)
 |-- laureate_id: string (nullable = true)
 |-- laureate_type: string (nullable = true)
 |-- full_name: string (nullable = true)
 |-- birth_date: string (nullable = true)
 |-- birth_city: string (nullable = true)
 |-- birth_country: string (nullable = true)
 |-- sex: string (nullable = true)
 |-- organization_name: string (nullable = true)
 |-- organization_city: string (nullable = true)
 |-- organization_country: string (nullable = true)
 |-- death_date: string (nullable = true)
 |-- death_city: string (nullable = true)
 |-- death_country: string (nullable = true)



In [19]:
df_nobel.count()

911

In [17]:
df_nobel.groupBy("category").count().show()

+----------+-----+
|  category|count|
+----------+-----+
| Chemistry|  175|
|  Medicine|  211|
|   Physics|  204|
|Literature|  113|
| Economics|   78|
|     Peace|  130|
+----------+-----+



In [18]:
df_nobel.groupBy("birth_country").count().show()

+--------------------+-----+
|       birth_country|count|
+--------------------+-----+
|British Protector...|    1|
|          1875-06-06|    1|
|           Dabrovica|    1|
|              Russia|   13|
|          1905-06-21|    1|
|          1911-06-13|    1|
|    Germany (Poland)|    7|
|               Yemen|    1|
|W&uuml;rttemberg ...|    1|
|           Neuchâtel|    1|
|          1927-03-06|    1|
|              Prague|    1|
|              Sweden|   23|
|          1937-06-23|    1|
|                 Ulm|    1|
|               Kyoto|    2|
|          Yakima, WA|    1|
|          1918-10-08|    1|
|            Azinhaga|    1|
|   Tomas Tranströmer|    1|
+--------------------+-----+
only showing top 20 rows



In [8]:
df_nobel.filter(df_nobel.sex == "Female").select("full_name").show(5, truncate = False)

+--------------------------------------------------------------------------------------+
|full_name                                                                             |
+--------------------------------------------------------------------------------------+
|Marie Curie, née Sklodowska                                                           |
|Baroness Bertha Sophie Felicita von Suttner, née Countess Kinsky von Chinic und Tettau|
|Grazia Deledda                                                                        |
|Sigrid Undset                                                                         |
|Jane Addams                                                                           |
+--------------------------------------------------------------------------------------+
only showing top 5 rows



In [9]:
df_nobel.filter(df_nobel.birth_country == "United States of America").select("full_name").show(5, truncate = False)

+-------------------------+
|full_name                |
+-------------------------+
|Theodore Roosevelt       |
|Elihu Root               |
|Theodore William Richards|
|Thomas Woodrow Wilson    |
|Robert Andrews Millikan  |
+-------------------------+
only showing top 5 rows



In [30]:
df_nobel.groupBy("full_name").count().filter("count > 2").show()
#.map(lambda x: len(x) > 2).show()

+--------------------+-----+
|           full_name|count|
+--------------------+-----+
|          Individual|  138|
|Comité internatio...|    3|
|                 1/3|    3|
|                 1/1|    7|
+--------------------+-----+



In [10]:
df_cosmetics = spark.read.csv(dir+cosmetics, header=True, inferSchema=True)
df_cosmetics.printSchema()

root
 |-- Label: string (nullable = true)
 |-- Brand: string (nullable = true)
 |-- Name: string (nullable = true)
 |-- Price: integer (nullable = true)
 |-- Rank: double (nullable = true)
 |-- Ingredients: string (nullable = true)
 |-- Combination: integer (nullable = true)
 |-- Dry: integer (nullable = true)
 |-- Normal: integer (nullable = true)
 |-- Oily: integer (nullable = true)
 |-- Sensitive: integer (nullable = true)



In [31]:
df_cosmetics.groupBy("Label").count().show()

+-----------+-----+
|      Label|count|
+-----------+-----+
|  Face Mask|  266|
|Moisturizer|  298|
|  Eye cream|  209|
|Sun protect|  170|
|  Treatment|  248|
|   Cleanser|  281|
+-----------+-----+



In [11]:
moisturizer = df_cosmetics.filter(df_cosmetics.Label == "Moisturizer").select("Brand","Name","Price")

In [12]:
moisturizer.show(7, truncate = False)

+--------------+---------------------------------------------+-----+
|Brand         |Name                                         |Price|
+--------------+---------------------------------------------+-----+
|LA MER        |Crème de la Mer                              |175  |
|SK-II         |Facial Treatment Essence                     |179  |
|DRUNK ELEPHANT|Protini™ Polypeptide Cream                   |68   |
|LA MER        |The Moisturizing Soft Cream                  |175  |
|IT COSMETICS  |Your Skin But Better™ CC+™ Cream with SPF 50+|38   |
|TATCHA        |The Water Cream                              |68   |
|DRUNK ELEPHANT|Lala Retro™ Whipped Cream                    |60   |
+--------------+---------------------------------------------+-----+
only showing top 7 rows



In [29]:
moisturizer.groupBy("Brand").agg(count("Name").alias("Count"), sum("Price").alias("Sum Price")) \
                                 .orderBy("Sum Price", ascending = False).show()

+------------------+-----+---------+
|             Brand|Count|Sum Price|
+------------------+-----+---------+
|            LA MER|   10|     2040|
|             SK-II|    8|     1512|
|          SHISEIDO|   18|     1476|
|             FRESH|   12|     1043|
|           LANCÔME|    9|      873|
|      ESTÉE LAUDER|    9|      778|
|              DIOR|    6|      772|
|      AMOREPACIFIC|    6|      700|
|            TATCHA|    8|      669|
|          CLINIQUE|   16|      628|
|          CAUDALIE|    9|      588|
|           ORIGINS|   10|      448|
|          ALGENIST|    5|      427|
|             MURAD|    7|      408|
|         HERBIVORE|    7|      401|
|      SUNDAY RILEY|    5|      387|
|   KATE SOMERVILLE|    5|      375|
|      IT COSMETICS|    9|      373|
|KIEHL'S SINCE 1851|    9|      358|
|           CLARINS|    4|      331|
+------------------+-----+---------+
only showing top 20 rows



In [80]:
moisturizer = moisturizer.withColumn("Rank", rank().over(Window.partitionBy("Brand").orderBy("Price")))

In [81]:
moisturizer = moisturizer.withColumn("Dense Rank", dense_rank().over(Window.partitionBy("Brand").orderBy("Price")))

In [82]:
moisturizer = moisturizer.withColumn("first", first("Price").over(Window.partitionBy("Brand")))

In [84]:
moisturizer = moisturizer.withColumn("last", last("Price").over(Window.partitionBy("Brand")))

In [85]:
moisturizer = moisturizer.withColumn("avg", avg("Price").over(Window.partitionBy("Brand")))

In [86]:
moisturizer = moisturizer.withColumn("sum", sum("Price").over(Window.partitionBy("Brand")))

In [87]:
moisturizer = moisturizer.withColumn("cume sum", sum("Price") \
                                     .over(Window.partitionBy("Brand").orderBy("Price").rowsBetween(Window.unboundedPreceding, 0)))

In [88]:
moisturizer = moisturizer.withColumn("Lead", lead("Price", 1).over(Window.partitionBy("Brand").orderBy("Price")))

In [89]:
moisturizer = moisturizer.withColumn("Lag", lag("Price", 1).over(Window.partitionBy("Brand").orderBy("Price")))

In [90]:
moisturizer.show(29)

+------------------+--------------------+-----+----+----------+-----+----+------------------+---+--------+----+----+
|             Brand|                Name|Price|Rank|Dense Rank|first|last|               avg|sum|cume sum|Lead| Lag|
+------------------+--------------------+-----+----+----------+-----+----+------------------+---+--------+----+----+
|           FARSÁLI|Volcanic Elixir P...|   39|   1|         1|   39|  54|              49.0|147|      39|  54|null|
|           FARSÁLI|Unicorn Essence A...|   54|   2|         2|   39|  54|              49.0|147|      93|  54|  39|
|           FARSÁLI|Rose Gold Elixir ...|   54|   2|         2|   39|  54|              49.0|147|     147|null|  54|
|          SMASHBOX|Photo Finish Prim...|   32|   1|         1|   32|  42|              39.5|158|      32|  42|null|
|          SMASHBOX|Camera Ready BB C...|   42|   2|         2|   32|  42|              39.5|158|      74|  42|  32|
|          SMASHBOX|Photo Finish Prim...|   42|   2|         2| 

In [13]:
df_cosmetics.filter(df_cosmetics.Label == "Moisturizer").filter(df_cosmetics.Dry == 1) \
    .select("Ingredients").show(5, truncate = False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [14]:
df_cosmetics.filter((df_cosmetics.Label == "Moisturizer") & (df_cosmetics.Dry == 1)) \
    .select("Ingredients").show(5, truncate = False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [66]:
df_disney = spark.read.csv(dir+disney, header=True, inferSchema=True)
df_disney.printSchema()

root
 |-- movie_title: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- genre: string (nullable = true)
 |-- mpaa_rating: string (nullable = true)
 |-- total_gross: integer (nullable = true)
 |-- inflation_adjusted_gross: long (nullable = true)



In [67]:
df_disney.groupBy(year("release_date")).count().show()

+------------------+-----+
|year(release_date)|count|
+------------------+-----+
|              1959|    1|
|              1990|   15|
|              1975|    1|
|              1977|    4|
|              2003|   19|
|              2007|   14|
|              2015|   11|
|              1955|    1|
|              2006|   19|
|              1961|    3|
|              2013|   11|
|              1988|   12|
|              1997|   23|
|              1994|   30|
|              1968|    1|
|              2014|   12|
|              1979|    1|
|              1946|    1|
|              1971|    1|
|              1950|    1|
+------------------+-----+
only showing top 20 rows



In [68]:
df_disney.groupBy(year("release_date").alias("Year")).sum("total_gross").show()

+----+----------------+
|Year|sum(total_gross)|
+----+----------------+
|1959|         9464608|
|1990|       613857056|
|1975|        31916500|
|1977|       102717599|
|2003|      1564114393|
|2007|      1436787754|
|2015|      2495662696|
|1955|        93600000|
|2006|      1427356974|
|1961|       188599723|
|2013|      1821352070|
|1988|       479129298|
|1997|       803617066|
|1994|      1092255030|
|1968|        21540050|
|2014|      1514179473|
|1979|        35841901|
|1946|        65000000|
|1971|        17871174|
|1950|        85000000|
+----+----------------+
only showing top 20 rows



In [75]:
df_disney.groupBy(year("release_date").alias("Year")) \
    .agg(sum("total_gross").alias("Sum Gross")).orderBy("Sum Gross", ascending=False).show(11)

+----+----------+
|Year| Sum Gross|
+----+----------+
|2016|2873393105|
|2015|2495662696|
|2013|1821352070|
|2003|1564114393|
|2010|1518975880|
|2014|1514179473|
|2012|1452972057|
|2007|1436787754|
|2006|1427356974|
|1998|1229279167|
|2009|1215142753|
+----+----------+
only showing top 11 rows



In [71]:
df_disney.groupBy(month("release_date").alias("Month")) \
    .agg(sum("total_gross").alias("Sum Gross")).orderBy("Sum Gross", ascending=False).show(7)

+-----+----------+
|Month| Sum Gross|
+-----+----------+
|   11|6710416094|
|    6|5720518013|
|    5|4937909068|
|   12|4350869253|
|    3|3148169471|
|    7|3032210005|
|    8|2568209698|
+-----+----------+
only showing top 7 rows



In [74]:
df_disney.groupBy(dayofmonth("release_date").alias("Day")) \
    .agg(sum("total_gross").alias("Sum Gross")).orderBy("Sum Gross", ascending=False).show(7)

+---+----------+
|Day| Sum Gross|
+---+----------+
| 18|2052760281|
| 22|1987565267|
|  4|1946219991|
| 25|1763384345|
|  1|1713458929|
|  9|1616935261|
| 21|1601766026|
+---+----------+
only showing top 7 rows



In [22]:
df_yearly_deaths = spark.read.csv(dir + yearly_deaths, header=True, inferSchema=True)
df_yearly_deaths.printSchema()

root
 |-- year: integer (nullable = true)
 |-- births: integer (nullable = true)
 |-- deaths: integer (nullable = true)
 |-- clinic: string (nullable = true)



In [27]:
df_yearly_deaths.groupBy("year").agg(sum("births").alias("births"), sum("deaths").alias("deaths")) \
    .orderBy("deaths", ascending = False).show(7)

+----+------+------+
|year|births|deaths|
+----+------+------+
|1842|  5946|   720|
|1846|  7764|   564|
|1843|  5799|   438|
|1844|  6113|   328|
|1841|  5478|   323|
|1845|  6733|   307|
+----+------+------+



In [35]:
df_yearly_deaths.withColumn("proportion_deaths", col("deaths")/col("births")) \
    .orderBy("proportion_deaths", ascending = False).show(7)

+----+------+------+--------+-------------------+
|year|births|deaths|  clinic|  proportion_deaths|
+----+------+------+--------+-------------------+
|1842|  3287|   518|clinic 1|0.15759050806206268|
|1846|  4010|   459|clinic 1| 0.1144638403990025|
|1843|  3060|   274|clinic 1|0.08954248366013072|
|1844|  3157|   260|clinic 1|0.08235666772252138|
|1841|  3036|   237|clinic 1|0.07806324110671936|
|1842|  2659|   202|clinic 2| 0.0759684091763821|
|1845|  3492|   241|clinic 1|0.06901489117983964|
+----+------+------+--------+-------------------+
only showing top 7 rows



In [23]:
df_monthly_deaths = spark.read.csv(dir + monthly_deaths, header=True, inferSchema=True)
df_monthly_deaths.printSchema()

root
 |-- date: string (nullable = true)
 |-- births: integer (nullable = true)
 |-- deaths: integer (nullable = true)



In [31]:
df_monthly_deaths.withColumn("proportion_deaths", col("deaths")/col("births")) \
    .orderBy("proportion_deaths", ascending = False).show(7)

+----------+------+------+-------------------+
|      date|births|deaths|  proportion_deaths|
+----------+------+------+-------------------+
|1842-12-01|   239|    75| 0.3138075313807531|
|1842-10-01|   242|    71|0.29338842975206614|
|1842-08-01|   216|    55|0.25462962962962965|
|1842-11-01|   209|    48|0.22966507177033493|
|1841-11-01|   235|    53|  0.225531914893617|
|1842-01-01|   307|    64|0.20846905537459284|
|1842-07-01|   231|    48| 0.2077922077922078|
+----------+------+------+-------------------+
only showing top 7 rows



In [37]:
df_colors = spark.read.csv(dir + colors, header=True, inferSchema=True)
df_colors.printSchema()

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- rgb: string (nullable = true)
 |-- is_trans: string (nullable = true)



In [47]:
df_colors.count()

135

In [46]:
df_colors.select("name").distinct().count()

135

In [50]:
df_colors.groupBy("is_trans").agg(count("id"), count("rgb"), count("name")).show()

+--------+---------+----------+-----------+
|is_trans|count(id)|count(rgb)|count(name)|
+--------+---------+----------+-----------+
|       f|      107|       107|        107|
|       t|       28|        28|         28|
+--------+---------+----------+-----------+



In [6]:
df_sets = spark.read.csv(dir + sets, header=True, inferSchema=True)
df_sets.printSchema()

root
 |-- set_num: string (nullable = true)
 |-- name: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- theme_id: integer (nullable = true)
 |-- num_parts: integer (nullable = true)



In [7]:
df_sets.show(7)

+-------+--------------------+----+--------+---------+
|set_num|                name|year|theme_id|num_parts|
+-------+--------------------+----+--------+---------+
|   00-1|     Weetabix Castle|1970|     414|      471|
| 0011-2|   Town Mini-Figures|1978|      84|       12|
| 0011-3|Castle 2 for 1 Bo...|1987|     199|        2|
| 0012-1|  Space Mini-Figures|1979|     143|       12|
| 0013-1|  Space Mini-Figures|1979|     143|       12|
| 0014-1|  Space Mini-Figures|1979|     143|       12|
| 0015-1|  Space Mini-Figures|1979|     143|       18|
+-------+--------------------+----+--------+---------+
only showing top 7 rows



In [54]:
df_sets.groupBy("year").agg(count("theme_id").alias("theme_id")).orderBy("theme_id", ascending = False).show()

+----+--------+
|year|theme_id|
+----+--------+
|2014|     715|
|2015|     670|
|2012|     615|
|2016|     609|
|2013|     593|
|2011|     502|
|2017|     470|
|2002|     447|
|2010|     444|
|2003|     415|
|2009|     403|
|2004|     371|
|2008|     349|
|2001|     339|
|2005|     330|
|2000|     327|
|1998|     325|
|2007|     319|
|1999|     300|
|2006|     283|
+----+--------+
only showing top 20 rows



In [92]:
df_movies = spark.read.csv(dir + movies, header=True, inferSchema=True)
df_movies.printSchema()

root
 |-- rank: string (nullable = true)
 |-- title: string (nullable = true)
 |-- genre: string (nullable = true)
 |-- wiki_plot: string (nullable = true)
 |-- imdb_plot: string (nullable = true)



In [99]:
df_movies = df_movies.withColumn("title split", split("title", " "))

In [106]:
df_movies = df_movies.withColumn("len title", size("title split"))

In [110]:
df_movies.filter(col("len title") > 2).select("title").show(7, truncate = False)

+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|title                                                                                                                                                                                                                                                                                                      |
+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
| drug baron Virgil ""The Turk"" Sollozzo                                                     

In [114]:
df_keywords = spark.read.csv(dir + keywords, header=True, inferSchema=True)
df_keywords.printSchema()

root
 |-- Ad Group: string (nullable = true)
 |-- Keyword: string (nullable = true)
 |-- Campaign: string (nullable = true)
 |-- Criterion Type: string (nullable = true)



In [117]:
df_keywords.filter(col("Criterion Type") == "Exact").groupBy("Ad Group", "Criterion Type") \
    .agg(count("Keyword")).show(7)

+-----------------+--------------+--------------+
|         Ad Group|Criterion Type|count(Keyword)|
+-----------------+--------------+--------------+
|        sofa beds|         Exact|            12|
|        recliners|         Exact|            12|
|convertible sofas|         Exact|            12|
|       love seats|         Exact|            12|
|            sofas|         Exact|            12|
+-----------------+--------------+--------------+



In [120]:
df_transfusion = spark.read.csv(dir + transfusion, header=True, inferSchema=True)
df_transfusion.printSchema()

root
 |-- Recency (months): double (nullable = true)
 |-- Frequency (times): integer (nullable = true)
 |-- Monetary (c.c. blood): integer (nullable = true)
 |-- Time (months): double (nullable = true)
 |-- whether he/she donated blood in March 2007: integer (nullable = true)



In [121]:
df_transfusion = df_transfusion.withColumnRenamed("whether he/she donated blood in March 2007", "target")

In [122]:
df_transfusion.printSchema()

root
 |-- Recency (months): double (nullable = true)
 |-- Frequency (times): integer (nullable = true)
 |-- Monetary (c.c. blood): integer (nullable = true)
 |-- Time (months): double (nullable = true)
 |-- target: integer (nullable = true)



In [125]:
df_temperature = spark.read.csv(dir + temperature, header=True, inferSchema=True)
df_temperature.printSchema()

root
 |-- year: integer (nullable = true)
 |-- degrees_celsius: double (nullable = true)



In [127]:
df_cookie = spark.read.csv(dir + cookie, header=True, inferSchema=True)
df_cookie.printSchema()

root
 |-- userid: integer (nullable = true)
 |-- version: string (nullable = true)
 |-- sum_gamerounds: integer (nullable = true)
 |-- retention_1: boolean (nullable = true)
 |-- retention_7: boolean (nullable = true)



In [134]:
df_cookie.groupBy("version").agg(count("retention_1"), count("retention_7")).show()

+-------+------------------+------------------+
|version|count(retention_1)|count(retention_7)|
+-------+------------------+------------------+
|gate_30|             44700|             44700|
|gate_40|             45489|             45489|
+-------+------------------+------------------+



In [138]:
df_cookie.groupBy("version", "retention_1", "retention_7") \
    .agg(count("userid").alias("Count"), sum("sum_gamerounds").alias("GameRounds")) \
    .orderBy("GameRounds", ascending = False).show()

+-------+-----------+-----------+-----+----------+
|version|retention_1|retention_7|Count|GameRounds|
+-------+-----------+-----------+-----+----------+
|gate_40|       true|       true| 6506|   1237977|
|gate_30|       true|       true| 6676|   1227625|
|gate_40|       true|      false|13613|    680997|
|gate_30|       true|      false|13358|    663819|
|gate_40|      false|      false|23597|    281118|
|gate_30|      false|      false|22840|    269963|
|gate_30|      false|       true| 1826|    183388|
|gate_40|      false|       true| 1773|    133438|
+-------+-----------+-----------+-----+----------+



In [16]:
df_cc = spark.read.csv(dir + cc, header=True, inferSchema=True)
df_cc.printSchema()

root
 |-- b: string (nullable = true)
 |-- 30.83: string (nullable = true)
 |-- 02: double (nullable = true)
 |-- u: string (nullable = true)
 |-- g4: string (nullable = true)
 |-- w: string (nullable = true)
 |-- v: string (nullable = true)
 |-- 1.25: double (nullable = true)
 |-- t8: string (nullable = true)
 |-- t9: string (nullable = true)
 |-- 01: integer (nullable = true)
 |-- f: string (nullable = true)
 |-- g12: string (nullable = true)
 |-- 00202: string (nullable = true)
 |-- 014: integer (nullable = true)
 |-- +: string (nullable = true)



In [17]:
df_cc.isnull().sum()

AttributeError: 'DataFrame' object has no attribute 'isnull'

In [20]:
df_social_us = spark.read.json(dir + social_us)
df_social_us.printSchema()

root
 |-- as_of: string (nullable = true)
 |-- created_at: string (nullable = true)
 |-- locations: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- woeid: long (nullable = true)
 |-- trends: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- promoted_content: string (nullable = true)
 |    |    |-- query: string (nullable = true)
 |    |    |-- tweet_volume: long (nullable = true)
 |    |    |-- url: string (nullable = true)



In [21]:
df_social_ww = spark.read.json(dir + social_ww)
df_social_ww.printSchema()

root
 |-- as_of: string (nullable = true)
 |-- created_at: string (nullable = true)
 |-- locations: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- woeid: long (nullable = true)
 |-- trends: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- promoted_content: string (nullable = true)
 |    |    |-- query: string (nullable = true)
 |    |    |-- tweet_volume: long (nullable = true)
 |    |    |-- url: string (nullable = true)



In [23]:
df_dates = spark.read.csv(dir + dates, header=True, inferSchema=True)
df_dates.printSchema()

root
 |-- order: integer (nullable = true)
 |-- date: string (nullable = true)
 |-- attacks: integer (nullable = true)
 |-- deaths: integer (nullable = true)



In [24]:
df_dates.show(7)

+-----+----------+-------+------+
|order|      date|attacks|deaths|
+-----+----------+-------+------+
|    1|1854-08-19|      1|     1|
|    2|1854-08-20|      1|     0|
|    3|1854-08-21|      1|     2|
|    4|1854-08-22|      0|     0|
|    5|1854-08-23|      1|     0|
|    6|1854-08-24|      1|     2|
|    7|1854-08-25|      0|     0|
+-----+----------+-------+------+
only showing top 7 rows



In [28]:
df_dates.groupBy(month("date").alias("Month"), dayofmonth("date").alias("Day")) \
    .agg(sum("attacks").alias("Attacks"), sum("deaths").alias("deaths")) \
    .orderBy("deaths", ascending = False).show(31)

+-----+---+-------+------+
|Month|Day|Attacks|deaths|
+-----+---+-------+------+
|    9|  2|    116|   127|
|    9|  3|     54|    76|
|    9|  4|     46|    71|
|    9|  1|    143|    70|
|    9|  5|     36|    45|
|    9|  6|     20|    37|
|    9|  7|     28|    32|
|    9|  8|     12|    30|
|    9|  9|     11|    24|
|    9| 10|      5|    18|
|    9| 11|      5|    15|
|    9| 13|      3|    13|
|    9| 15|      1|     8|
|    9| 16|      4|     6|
|    9| 14|      0|     6|
|    9| 12|      1|     6|
|    9| 17|      2|     5|
|    9| 19|      0|     3|
|    9| 23|      1|     3|
|    8| 31|     56|     3|
|    9| 26|      1|     2|
|    9| 22|      1|     2|
|    9| 18|      3|     2|
|    8| 21|      1|     2|
|    8| 30|      8|     2|
|    9| 28|      0|     2|
|    8| 24|      1|     2|
|    8| 19|      1|     1|
|    8| 27|      1|     1|
|    8| 29|      1|     1|
|    9| 29|      0|     1|
+-----+---+-------+------+
only showing top 31 rows



In [29]:
df_deaths = spark.read.csv(dir + deaths, header=True, inferSchema=True)
df_deaths.printSchema()

root
 |-- Death: integer (nullable = true)
 |-- X coordinate: double (nullable = true)
 |-- Y coordinate: double (nullable = true)



In [30]:
df_pumps = spark.read.csv(dir + pumps, header=True, inferSchema=True)
df_pumps.printSchema()

root
 |-- Pump Name: string (nullable = true)
 |-- X coordinate: double (nullable = true)
 |-- Y coordinate: double (nullable = true)



In [32]:
df_pumps_deaths = spark.read.csv(dir + pumps_deaths, header=True, inferSchema=True)
df_pumps_deaths.printSchema()

root
 |-- Number of deaths: integer (nullable = true)
 |-- X coordinate: double (nullable = true)
 |-- Y coordinate: double (nullable = true)



In [76]:
df_benchmark = spark.read.csv(dir + benchmark, header=True, inferSchema=True)
df_benchmark.printSchema()

root
 |-- Date: string (nullable = true)
 |-- S&P 500: double (nullable = true)



In [39]:
df_benchmark.show(7)

+----------+-------+
|      Date|S&P 500|
+----------+-------+
|2016-01-01|   null|
|2016-01-04|2012.66|
|2016-01-05|2016.71|
|2016-01-06|1990.26|
|2016-01-07|1943.09|
|2016-01-08|1922.03|
|2016-01-11|1923.67|
+----------+-------+
only showing top 7 rows



In [54]:
df_benchmark.groupBy(month("Date").alias("Month")).agg(max("S&P 500").alias("Max S&P")) \
    .orderBy("Max S&P", ascending = False).show()

+-----+-------+
|Month|Max S&P|
+-----+-------+
|   12|2271.72|
|   11|2213.35|
|    8|2190.15|
|    9|2186.48|
|    7|2175.03|
|   10|2163.66|
|    6|2119.12|
|    4| 2102.4|
|    5|2099.06|
|    3|2063.95|
|    1|2016.71|
|    2| 1951.7|
+-----+-------+



In [77]:
df_benchmark = df_benchmark.withColumn("Max S&P", max("S&P 500").over(Window.partitionBy(month("Date")).orderBy("Date")))

In [78]:
df_benchmark = df_benchmark.withColumn("Max M", max("S&P 500").over(Window.partitionBy(month("Date"))))

In [79]:
df_benchmark = df_benchmark.withColumn("Min S&P", min("S&P 500").over(Window.partitionBy(month("Date")).orderBy("Date")))

In [80]:
df_benchmark = df_benchmark.withColumn("Min M", min("S&P 500").over(Window.partitionBy(month("Date"))))

In [81]:
df_benchmark = df_benchmark.withColumn("Avg S&P", avg("S&P 500").over(Window.partitionBy(month("Date")).orderBy("Date")))

In [82]:
df_benchmark = df_benchmark.withColumn("Avg M", avg("S&P 500").over(Window.partitionBy(month("Date"))))

In [70]:
df_benchmark = df_benchmark.withColumn("Count", count("S&P 500").over(Window.partitionBy(month("Date")).orderBy("Date")))

In [83]:
df_benchmark = df_benchmark.withColumn("Rank", rank().over(Window.partitionBy(month("Date")).orderBy("Date")))

In [72]:
df_benchmark = df_benchmark.withColumn("Dense Rank", dense_rank().over(Window.partitionBy(month("Date")).orderBy("Date")))

In [85]:
df_benchmark.sort("Date").show(31)

+----------+-------+-------+-------+-------+-------+------------------+------------------+----+
|      Date|S&P 500|Max S&P|  Max M|Min S&P|  Min M|           Avg S&P|             Avg M|Rank|
+----------+-------+-------+-------+-------+-------+------------------+------------------+----+
|2016-01-01|   null|   null|2016.71|   null|1859.33|              null|1918.5978947368426|   1|
|2016-01-04|2012.66|2012.66|2016.71|2012.66|1859.33|           2012.66|1918.5978947368426|   2|
|2016-01-05|2016.71|2016.71|2016.71|2012.66|1859.33|          2014.685|1918.5978947368426|   3|
|2016-01-06|1990.26|2016.71|2016.71|1990.26|1859.33|2006.5433333333333|1918.5978947368426|   4|
|2016-01-07|1943.09|2016.71|2016.71|1943.09|1859.33|           1990.68|1918.5978947368426|   5|
|2016-01-08|1922.03|2016.71|2016.71|1922.03|1859.33|           1976.95|1918.5978947368426|   6|
|2016-01-11|1923.67|2016.71|2016.71|1922.03|1859.33|           1968.07|1918.5978947368426|   7|
|2016-01-12|1938.68|2016.71|2016.71|1922

In [86]:
df_benchmark.filter(month("Date") == 2).sort("Date").show(31)

+----------+-------+-------+------+-------+-------+------------------+------------------+----+
|      Date|S&P 500|Max S&P| Max M|Min S&P|  Min M|           Avg S&P|             Avg M|Rank|
+----------+-------+-------+------+-------+-------+------------------+------------------+----+
|2016-02-01|1939.38|1939.38|1951.7|1939.38|1829.08|           1939.38|1904.4185000000002|   1|
|2016-02-02|1903.03|1939.38|1951.7|1903.03|1829.08|          1921.205|1904.4185000000002|   2|
|2016-02-03|1912.53|1939.38|1951.7|1903.03|1829.08|1918.3133333333333|1904.4185000000002|   3|
|2016-02-04|1915.45|1939.38|1951.7|1903.03|1829.08|1917.5974999999999|1904.4185000000002|   4|
|2016-02-05|1880.05|1939.38|1951.7|1880.05|1829.08|1910.0879999999997|1904.4185000000002|   5|
|2016-02-08|1853.44|1939.38|1951.7|1853.44|1829.08|1900.6466666666665|1904.4185000000002|   6|
|2016-02-09|1852.21|1939.38|1951.7|1852.21|1829.08|1893.7271428571428|1904.4185000000002|   7|
|2016-02-10|1851.86|1939.38|1951.7|1851.86|1829.08

In [35]:
df_stock = spark.read.csv(dir + stock, header=True, inferSchema=True)
df_stock.printSchema()

root
 |-- Date: string (nullable = true)
 |-- Amazon: double (nullable = true)
 |-- Facebook: double (nullable = true)



In [40]:
df_stock.show(7)

+----------+-----------------+-----------------+
|      Date|           Amazon|         Facebook|
+----------+-----------------+-----------------+
|2016-01-04|        636.98999|       102.220001|
|2016-01-05|       633.789978|       102.730003|
|2016-01-06|       632.650024|       102.970001|
|2016-01-07|607.9400019999999|97.91999799999999|
|2016-01-08|       607.049988|97.33000200000001|
|2016-01-11|        617.73999|        97.510002|
|2016-01-12|617.8900150000001|        99.370003|
+----------+-----------------+-----------------+
only showing top 7 rows



In [95]:
df_stock = df_stock.select("Date","Amazon")

In [97]:
df_stock = df_stock.withColumn("Max", max("Amazon").over(Window.partitionBy(month("Date")).orderBy("Date")))

In [98]:
df_stock = df_stock.withColumn("Max M", max("Amazon").over(Window.partitionBy(month("Date"))))

In [99]:
df_stock = df_stock.withColumn("Min", min("Amazon").over(Window.partitionBy(month("Date")).orderBy("Date")))

In [100]:
df_stock = df_stock.withColumn("Min M", min("Amazon").over(Window.partitionBy(month("Date"))))

In [101]:
df_stock.show(31)

+----------+-----------------+-----------------+----------+-----------------+----------+
|      Date|           Amazon|              Max|     Max M|              Min|     Min M|
+----------+-----------------+-----------------+----------+-----------------+----------+
|2016-12-01|       743.650024|       743.650024|774.340027|       743.650024|740.340027|
|2016-12-02|       740.340027|       743.650024|774.340027|       740.340027|740.340027|
|2016-12-05|759.3599849999999|759.3599849999999|774.340027|       740.340027|740.340027|
|2016-12-06|       764.719971|       764.719971|774.340027|       740.340027|740.340027|
|2016-12-07|       770.419983|       770.419983|774.340027|       740.340027|740.340027|
|2016-12-08|       767.330017|       770.419983|774.340027|       740.340027|740.340027|
|2016-12-09|       768.659973|       770.419983|774.340027|       740.340027|740.340027|
|2016-12-12|       760.119995|       770.419983|774.340027|       740.340027|740.340027|
|2016-12-13|       77

In [38]:
df_iris = spark.read.csv(dir + iris, header=True, inferSchema=True)
df_iris.printSchema()

root
 |-- sepal_length: double (nullable = true)
 |-- sepal_width: double (nullable = true)
 |-- petal_length: double (nullable = true)
 |-- petal_width: double (nullable = true)
 |-- species: string (nullable = true)



In [103]:
df_apps = spark.read.csv(dir + apps, header=True, inferSchema=True)
df_apps.printSchema()

root
 |-- _c0: integer (nullable = true)
 |-- App: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Rating: string (nullable = true)
 |-- Reviews: string (nullable = true)
 |-- Size: string (nullable = true)
 |-- Installs: string (nullable = true)
 |-- Type: string (nullable = true)
 |-- Price: string (nullable = true)
 |-- Content Rating: string (nullable = true)
 |-- Genres: string (nullable = true)
 |-- Last Updated: string (nullable = true)
 |-- Current Ver: string (nullable = true)
 |-- Android Ver: string (nullable = true)



In [108]:
df_apps.show(3)

+---+--------------------+--------------+------+-------+----+----------+----+-----+--------------+--------------------+----------------+-----------+------------+
|_c0|                 App|      Category|Rating|Reviews|Size|  Installs|Type|Price|Content Rating|              Genres|    Last Updated|Current Ver| Android Ver|
+---+--------------------+--------------+------+-------+----+----------+----+-----+--------------+--------------------+----------------+-----------+------------+
|  0|Photo Editor & Ca...|ART_AND_DESIGN|   4.1|    159|19.0|   10,000+|Free|    0|      Everyone|        Art & Design| January 7, 2018|      1.0.0|4.0.3 and up|
|  1| Coloring book moana|ART_AND_DESIGN|   3.9|    967|14.0|  500,000+|Free|    0|      Everyone|Art & Design;Pret...|January 15, 2018|      2.0.0|4.0.3 and up|
|  2|U Launcher Lite –...|ART_AND_DESIGN|   4.7|  87510| 8.7|5,000,000+|Free|    0|      Everyone|        Art & Design|  August 1, 2018|      1.2.4|4.0.3 and up|
+---+--------------------+--

In [104]:
df_reviews = spark.read.csv(dir + reviews, header=True, inferSchema=True)
df_reviews.printSchema()

root
 |-- App: string (nullable = true)
 |-- Translated_Review: string (nullable = true)
 |-- Sentiment: string (nullable = true)
 |-- Sentiment_Polarity: string (nullable = true)
 |-- Sentiment_Subjectivity: string (nullable = true)



In [107]:
df_reviews.show(3)

+--------------------+--------------------+--------------------+------------------+----------------------+
|                 App|   Translated_Review|           Sentiment|Sentiment_Polarity|Sentiment_Subjectivity|
+--------------------+--------------------+--------------------+------------------+----------------------+
|10 Best Foods for...|"I like eat delic...| also ""Best Befo...|          Positive|                   1.0|
|10 Best Foods for...|This help eating ...|            Positive|              0.25|   0.28846153846153844|
|10 Best Foods for...|                 nan|                 nan|               nan|                   nan|
+--------------------+--------------------+--------------------+------------------+----------------------+
only showing top 3 rows



In [117]:
df_reviews.select('Sentiment_Subjectivity').distinct().show(truncate = False)

+--------------------------------------------------------------------------------------------+
|Sentiment_Subjectivity                                                                      |
+--------------------------------------------------------------------------------------------+
|0.32222222222222224                                                                         |
|0.5527777777777778                                                                          |
|0.5272727272727272                                                                          |
| keep up. Perfection."                                                                      |
|0.7077777777777777                                                                          |
|0.503030303030303                                                                           |
|0.5738095238095238                                                                          |
| currencies good idea""^^ A Currencies easly adde

In [111]:
df_joined = df_apps.join(df_reviews, on = 'App', how = 'inner').select('App', 'Category', 'Rating', 'Reviews', 'Type', 'Price', 'Genres')

In [122]:
df_joined.explain()

== Physical Plan ==
*(2) Project [App#10188, Category#10189, Rating#10190, Reviews#10191, Type#10194, Price#10195, Genres#10197]
+- *(2) BroadcastHashJoin [App#10188], [App#10231], Inner, BuildLeft, false
   :- BroadcastExchange HashedRelationBroadcastMode(List(input[0, string, false]),false), [id=#2219]
   :  +- *(1) Filter isnotnull(App#10188)
   :     +- FileScan csv [App#10188,Category#10189,Rating#10190,Reviews#10191,Type#10194,Price#10195,Genres#10197] Batched: false, DataFilters: [isnotnull(App#10188)], Format: CSV, Location: InMemoryFileIndex[file:/Users/abhinavmohanty/Documents/Python/DataCamp/Data Scientist/Projects/Th..., PartitionFilters: [], PushedFilters: [IsNotNull(App)], ReadSchema: struct<App:string,Category:string,Rating:string,Reviews:string,Type:string,Price:string,Genres:st...
   +- *(2) Filter isnotnull(App#10231)
      +- FileScan csv [App#10231] Batched: false, DataFilters: [isnotnull(App#10231)], Format: CSV, Location: InMemoryFileIndex[file:/Users/abhinavmohan

In [112]:
df_joined.show(7)

+--------------------+------------------+------+-------+----+-----+----------------+
|                 App|          Category|Rating|Reviews|Type|Price|          Genres|
+--------------------+------------------+------+-------+----+-----+----------------+
|10 Best Foods for...|HEALTH_AND_FITNESS|     4|   2490|Free|    0|Health & Fitness|
|10 Best Foods for...|HEALTH_AND_FITNESS|     4|   2490|Free|    0|Health & Fitness|
|10 Best Foods for...|HEALTH_AND_FITNESS|     4|   2490|Free|    0|Health & Fitness|
|10 Best Foods for...|HEALTH_AND_FITNESS|     4|   2490|Free|    0|Health & Fitness|
|10 Best Foods for...|HEALTH_AND_FITNESS|     4|   2490|Free|    0|Health & Fitness|
|10 Best Foods for...|HEALTH_AND_FITNESS|     4|   2490|Free|    0|Health & Fitness|
|10 Best Foods for...|HEALTH_AND_FITNESS|     4|   2490|Free|    0|Health & Fitness|
+--------------------+------------------+------+-------+----+-----+----------------+
only showing top 7 rows



In [121]:
df_joined.groupBy('Type').count().show()

+----+-----+
|Type|count|
+----+-----+
|Free|60642|
|Paid|  914|
+----+-----+

